# 🤖 LinkedIn Engagement Predictor — Model Training
**Author:** Solomon Uche

Train a model to predict post engagement rate from post features before publishing.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import xgboost as xgb
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1. Load & Feature Engineering

In [ ]:
df = pd.read_csv('../../data/raw/post_metrics.csv', parse_dates=['date_posted'])
published = df[df['impressions'] > 0].copy()

# Feature engineering
published['month'] = published['date_posted'].dt.month
published['week_num'] = published['date_posted'].dt.isocalendar().week
published['hashtag_density'] = published['hashtag_count'] / published['word_count'].clip(lower=1)
published['total_interactions'] = published['reactions'] + published['comments'] + published['shares'] + published['saves']
published['save_rate'] = published['saves'] / published['impressions'].clip(lower=1)
published['comment_rate'] = published['comments'] / published['impressions'].clip(lower=1)

print(f'Features created. Dataset: {published.shape}')
published[['word_count','hashtag_count','hashtag_density','has_cta','engagement_rate']].describe()

## 2. Encode Categorical Features

In [ ]:
# One-hot encode categoricals
cat_features = ['post_type','content_format','day_of_week','time_of_day','topic_category','hook_type']
df_encoded = pd.get_dummies(published, columns=cat_features, drop_first=True)

# Select features for model
feature_cols = ['word_count','hashtag_count','hashtag_density','has_cta','series_post','month'] + \
               [c for c in df_encoded.columns if any(c.startswith(cat) for cat in cat_features)]

X = df_encoded[feature_cols].fillna(0)
y = df_encoded['engagement_rate']

print(f'Feature matrix: {X.shape}')
print(f'Target: engagement_rate (mean={y.mean():.4f}, std={y.std():.4f})')

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

## 4. Train Multiple Models & Compare

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }
    print(f'{name}: MAE={results[name]["MAE"]:.4f}, R²={results[name]["R2"]:.3f}')

results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print('\n📊 Model Comparison:')
print(results_df)

## 5. Feature Importance (Best Model)

In [ ]:
best_model = models['Random Forest']  # Update based on results above

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
plt.title('Top 15 Most Important Features for Engagement Prediction', fontsize=13)
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.savefig('../../assets/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Model & Metrics

In [ ]:
# Save model
joblib.dump(best_model, '../../models/engagement_predictor_v1.pkl')

# Save metrics
metrics = {
    'model': 'RandomForestRegressor',
    'version': 'v1',
    'trained_on': str(pd.Timestamp.now().date()),
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'features': list(X.columns),
    'performance': results['Random Forest']
}
with open('../../models/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('✅ Model saved to models/engagement_predictor_v1.pkl')
print('✅ Metrics saved to models/model_metrics.json')

## 7. Predict New Post Engagement

In [ ]:
# Example: Predict engagement for a new post draft
# Fill in your post attributes below

new_post = {
    'word_count': 250,
    'hashtag_count': 7,
    'has_cta': True,
    'series_post': False,
    'post_type': 'Educational',
    'content_format': 'List',
    'day_of_week': 'Tuesday',
    'time_of_day': 'Morning',
    'topic_category': 'AI',
    'hook_type': 'Statement',
    'month': 3
}

print('📝 Post Attributes:')
for k, v in new_post.items():
    print(f'   {k}: {v}')

# NOTE: In production, this would be passed through the same preprocessing pipeline
# See analysis/scripts/engagement_predictor.py for the full inference pipeline
print('\n🔮 Run engagement_predictor.py to get predicted engagement rate for this post.')